In [ ]:
# Cell 1: Install arc-agi from competition wheels
import subprocess, sys, os
wheels = '/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels'
if os.path.exists(wheels):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-index', f'--find-links={wheels}', 'arc-agi'])
    print('[Cell 1] arc-agi installed from competition wheels')
else:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'arc-agi'])
    print('[Cell 1] arc-agi installed via pip')


In [ ]:
# Cell 2: VericodingAgent with one-hot D4 Zobrist hash
import os, random, collections, numpy as np

agent_code = '''import random, collections
import numpy as np
from .agent import Agent
from arcengine import GameAction

class VericodingAgent(Agent):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._step = 0
        self._graph = collections.defaultdict(dict)
        self._visited = set()
        self._last_hash = None
        self._last_key = None
        rng = np.random.RandomState(42)
        self._zob = rng.randint(0, 2**63-1, size=(64, 64, 16), dtype=np.uint64)

    def _hash(self, grid):
        best = np.uint64(2**64-1)
        for rot in range(4):
            for flip in (False, True):
                g = np.rot90(grid, rot)
                if flip: g = np.fliplr(g)
                gh, gw = g.shape[:2]
                # ONE-HOT: exact color channel per pixel (NOT broadcast)
                onehot = np.eye(16, dtype=np.uint64)[np.clip(g, 0, 15)]
                hv = int(np.bitwise_xor.reduce(self._zob[:gh, :gw, :] * onehot, axis=None))
                if hv < int(best): best = np.uint64(hv)
        return int(best)

    def _get_action_idx(self, action):
        if hasattr(action, 'action_type'): return action.action_type
        if hasattr(action, 'value'): return action.value
        return int(action)

    def choose_action(self, frames, latest_frame) -> GameAction:
        try:
            self._step += 1
            grid = np.array(latest_frame.frame[0], dtype=np.int32) if hasattr(latest_frame, 'frame') else np.zeros((2,2))
            w = grid.shape[1]
            raw = latest_frame.available_actions if hasattr(latest_frame, 'available_actions') else []
            avail = [a for a in raw]
            if not avail: return GameAction(1)

            cur_hash = self._hash(grid)
            self._visited.add(cur_hash)
            if self._last_hash is not None and self._last_key is not None:
                self._graph[self._last_hash][self._last_key] = cur_hash
            self._last_hash = cur_hash

            complex_a, simple_a = [], []
            for a in avail:
                idx = self._get_action_idx(a)
                if idx in [6, 7, 8] or (hasattr(a, 'is_complex') and a.is_complex()):
                    complex_a.append(idx)
                else:
                    simple_a.append(idx)

            nz = np.where(grid.flatten() > 0)[0]
            if complex_a and len(nz) > 0:
                pi = nz[self._step % len(nz)]
                y, x = divmod(int(pi), w)
                a = GameAction(complex_a[0])
                a.set_data({'x': int(x), 'y': int(y)})
                self._last_key = (complex_a[0], int(x), int(y))
                return a

            if simple_a:
                for act in simple_a:
                    if act not in self._graph.get(cur_hash, {}):
                        self._last_key = act
                        return GameAction(act)
                ch = simple_a[self._step % len(simple_a)]
                self._last_key = ch
                return GameAction(ch)

            return GameAction(1)
        except Exception:
            return GameAction(1)
'''

with open("/tmp/my_agent.py", "w") as f:
    f.write(agent_code)
print('[Cell 2] VericodingAgent written to /tmp/my_agent.py (one-hot D4 hash)')


In [ ]:
# Cell 3: Phase A/B orchestrator
import os, subprocess, shutil, sys, pandas as pd

if os.environ.get('KAGGLE_IS_COMPETITION_RERUN'):
    print('[Cell 3] Phase B: competition rerun detected')

    # Step 1: Wait for gateway sidecar
    subprocess.run(['curl', '--fail', '--retry', '100', '--retry-delay', '2', '--max-time', '5', 'http://gateway:8001/api/games'], check=True)

    # Step 2: Copy framework boilerplate
    src = '/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files'
    if os.path.isdir(src + '/agents'):
        shutil.copytree(src + '/agents', '/kaggle/working/agents', dirs_exist_ok=True)
    if os.path.exists(src + '/main.py'):
        shutil.copy(src + '/main.py', '/kaggle/working/main.py')

    # Step 3: Inject our agent
    shutil.copy('/tmp/my_agent.py', '/kaggle/working/agents/my_agent.py')

    # Step 4: Register agent with lowercase key
    with open('/kaggle/working/agents/__init__.py', 'w') as f:
        f.write('from .agent import Agent\nfrom .my_agent import VericodingAgent\nAVAILABLE_AGENTS = {"vericoding": VericodingAgent}\n')

    # Step 5: Run evaluation
    os.chdir('/kaggle/working')
    subprocess.run([sys.executable, 'main.py', '--agent', 'vericoding', '--output', '/kaggle/working/submission.parquet'], check=True)
    print('[Cell 3] Phase B complete')

else:
    print('[Cell 3] Phase A: writing dummy submission.parquet')
    pd.DataFrame(data=[['1_0', '1', True, 1]], columns=['row_id', 'game_id', 'end_of_game', 'score']).to_parquet('/kaggle/working/submission.parquet', index=False)
    print('[Cell 3] dummy parquet written -- click Submit to Competition')
